# Gauravi NSE Factor Dataset — factor IC in 20 lines

Reproduces a full cross-sectional **Information Coefficient** table for every
factor in the dataset, with an honestly-calibrated t-stat.

Requires `pandas`, `pyarrow`, `numpy`, `scipy`, and the dataset on disk (set
`GAURAVI_DATA_DIR`, or run this from the release folder).

**This is research data, not investment advice.**

In [ ]:
import numpy as np
import pandas as pd
import gauravi_data as gd

m = gd.manifest()
print(f"release {m['version']} | {m['n_symbols']} symbols | "
      f"{m['date_min']} -> {m['date_max']} | horizon {m['horizon_days']}d")
H = m["horizon_days"]

## 1. Load

`load_factors()` returns factor values, their per-date percentile ranks
(`rank_*`), the two composites, and `fwd_excess` — the forward `H`-day return
minus the benchmark.

`fwd_excess` contains future information by construction. It is the **target**;
never feed it in as a feature.

In [ ]:
f = gd.load_factors()
print(f.shape)
f[["date", "sym", "close", "rev", "deliv_pct", "earn_yield",
   "score_mr", "fwd_excess"]].tail()

## 2. Coverage first

Always look at this before believing an IC. `earn_growth` sits near 43% — a
factor measured on half the panel is not comparable to one measured on all of it.

In [ ]:
FACTORS = [c for c in f.columns if c.startswith("rank_")]
f[FACTORS].notna().mean().mul(100).round(1).sort_values(ascending=False)

## 3. The one statistical trap

Daily ICs computed against an `H`-day forward return **overlap**: today's
target shares `H-1` days with tomorrow's. Treating those as independent inflates
the t-stat by roughly `sqrt(H)` — about **5.5x** at `H=30`.

The fix below is deliberately crude and honest: keep only every `H`-th date, so
the remaining observations do not overlap at all.

In [ ]:
def ic_series(df, factor, target="fwd_excess"):
    """Spearman IC per date (needs >5 names to mean anything)."""
    sub = df[["date", factor, target]].dropna()
    return (sub.groupby("date")
               .apply(lambda g: g[factor].corr(g[target], method="spearman")
                      if len(g) > 5 else np.nan)
               .dropna())


def ic_table(df, factors, H):
    dates = np.sort(df["date"].unique())
    nonovlp = set(dates[::H])                  # non-overlapping rebalances only
    rows = []
    for col in factors:
        s = ic_series(df, col)
        if s.empty:
            continue
        ind = s[s.index.isin(nonovlp)]
        n = len(ind)
        t = ind.mean() / (ind.std(ddof=1) / np.sqrt(n)) if n > 2 else np.nan
        rows.append({"factor": col, "IC": s.mean(),
                     "t_naive": s.mean() / (s.std(ddof=1) / np.sqrt(len(s))),
                     "t_nonovlp": t, "n_indep": n})
    out = pd.DataFrame(rows).set_index("factor")
    out["significant"] = out["t_nonovlp"].abs() >= 1.96
    return out.reindex(out["t_nonovlp"].abs().sort_values(ascending=False).index)


tbl = ic_table(f, FACTORS + ["score_mr", "score_fund3"], H)
tbl.round(4)

## 4. How to read that table

Compare the `t_naive` and `t_nonovlp` columns. The naive column is the one most
published NSE factor studies report, and it is several times too large.

On our own build, **no factor clears `|t_nonovlp| >= 1.96`.** If you find one that
does, check its coverage in step 2 and how many independent observations
(`n_indep`) it rests on before getting excited — at 5 years and `H=30` there are
only ~41 of them, which is not many.

Also remember: the universe is *current* NIFTY-100 membership, so every number
here carries survivorship bias.

## 5. Quintile spread — the same signal, in return terms

IC is scale-free and hard to feel. This is the top-quintile minus
bottom-quintile forward excess return, on non-overlapping holds.

In [ ]:
def quintile_spread(df, score, H):
    d = df.dropna(subset=[score, "fwd_excess"]).copy()
    dates = np.sort(d["date"].unique())
    d = d[d["date"].isin(set(dates[::H]))]
    hi = d.groupby("date")[score].transform(lambda x: x.quantile(0.8))
    lo = d.groupby("date")[score].transform(lambda x: x.quantile(0.2))
    top, bot = d[d[score] >= hi], d[d[score] <= lo]
    hit = (top["fwd_excess"] > 0).mean()
    return pd.Series({
        "top_mean_%": top["fwd_excess"].mean() * 100,
        "bottom_mean_%": bot["fwd_excess"].mean() * 100,
        "spread_pp": (top["fwd_excess"].mean() - bot["fwd_excess"].mean()) * 100,
        "top_hit_rate_%": hit * 100,
        "hit_SE_pp": np.sqrt(hit * (1 - hit) / len(top)) * 100,
        "n_picks": len(top),
    })


pd.DataFrame({s: quintile_spread(f, s, H)
              for s in ("score_mr", "score_fund3", "rank_rev",
                        "rank_mom_12_1")}).T.round(2)

## 6. What we concluded — and what we did not

On the common sample, `score_mr` (momentum + reversal) at `H=30d` reaches a
top-quintile directional hit-rate around **52-53%** with a standard error near
**2pp**, against a ~51% coin-flip baseline. That is roughly one standard error from
55% and one from 51%: with only ~40 independent rebalances, the sample cannot tell
those apart. Treat every number in this notebook as a measurement with error bars,
not a result.

Delivery-based factors (`deliv_x_ret` and the `score_fund3` composite built on it)
looked better than this in an earlier build of the dataset. They were not — the
delivery source mislabelled each week's Friday as the following Sunday, silently
dropping one trading day per week. After that fix, they no longer beat plain
momentum+reversal, and on the common sample above they trail it.

Delivery % is in this dataset because it is **hard to get and cleanly aligned**,
not because we can show it predicts returns.

### Where to go next with this data

- Sector-neutralise the ranks before scoring (sector tags are on the roadmap).
- Try a fitted walk-forward combiner instead of equal weights on the ranks.
- Extend the history — `n_indep` is the binding constraint here, not the signal.
- Use `deliv_pct` for what delivery is actually informative about: separating
  genuine accumulation from intraday churn in your own screens.

In [ ]:
common = f.dropna(subset=["score_mr", "score_fund3"])
print(f"full: {len(f):,} rows -> common: {len(common):,} rows")
pd.DataFrame({s: quintile_spread(common, s, H)
              for s in ("score_mr", "score_fund3")}).T.round(2)

## 6. What we concluded — and what we did not

`score_mr` (momentum + reversal) measured a top-quintile directional hit-rate of
about **53%** at `H=30d` with a standard error near **2pp**, against a ~51%
coin-flip baseline. That is one standard error from 55% and one from 51%: the
sample cannot tell those apart.

Delivery-based factors (`deliv_x_ret` and the `score_fund3` composite that uses
it) looked better than this in an earlier build. They were not — the delivery
source mislabelled each week's Friday as the following Sunday, dropping one
trading day per week. After the fix, they no longer beat plain momentum+reversal.

Delivery % is in this dataset because it is **hard to get and cleanly aligned**,
not because we can show it predicts returns.

### Where to go next with this data

- Sector-neutralise the ranks before scoring (sector tags are on the roadmap).
- Try a fitted walk-forward combiner instead of equal weights on the ranks.
- Extend the horizon or history — `n_indep` is the binding constraint here, not
  the signal.
- Use `deliv_pct` for the thing delivery is actually informative about:
  distinguishing genuine accumulation from intraday churn in your own screens.